In [1]:
#import needed libraries
import pandas as pd
import numpy as np

In [2]:
#load  data rfm_based saved in desktop from SSMS
#may likely load without heaeders bcos of the way it was exported from ssms,
#if that is the case, you add the ending:--( ,header=none) and suppy the (--rfm.columns)

rfm = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\rfm_based.csv",header=None)
rfm.columns=['CustomerID','Recency','Frequency','Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,3,7,4310.00
2,12348.0,76,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,311,1,334.40


In [3]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4339 entries, 0 to 4338
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  4339 non-null   float64
 1   Recency     4339 non-null   int64  
 2   Frequency   4339 non-null   int64  
 3   Monetary    4339 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 135.7 KB


In [4]:
rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,4339.000000,4339.000000,4339.000000,4339.000000
mean,15299.936852,93.041484,4.271952,2048.215923
std,1721.889758,100.007757,7.705493,8984.248352
min,12346.000000,1.000000,1.000000,0.000000
25%,13812.500000,18.000000,1.000000,306.455000
50%,15299.000000,51.000000,2.000000,668.560000
75%,16778.500000,142.500000,5.000000,1660.315000
max,18287.000000,374.000000,210.000000,280206.020000


**Next step- Create RFM Scores Using Quartiles**  
we rank customers from 1-4

In [5]:
#Recency Score (lower Recency = better customer)
rfm['R_score'] = pd.qcut(rfm['Recency'],4, labels=[4,3,2,1])


In [7]:
#Frequency Score 
rfm['F_score']=pd.qcut(rfm['Frequency'].rank(method='first'),4,labels=[1,2,3,4])

In [8]:
#Monetary Score 
rfm['M_score']=pd.qcut(rfm['Monetary'],4,labels=[1,2,3,4])

**Next Step- Create RFM Score**

In [9]:
rfm['RFM_Score']=(
    rfm['R_score'].astype(str)+
    rfm['F_score'].astype(str)+
   rfm['M_score'].astype(str) )

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,12346.0,326,1,77183.60,1,1,4,114
1,12347.0,3,7,4310.00,4,4,4,444
2,12348.0,76,4,1797.24,2,3,4,234
3,12349.0,19,1,1757.55,3,1,4,314
4,12350.0,311,1,334.40,1,1,2,112


**Next Step- Create Customer Segments**
Now we classify customers.

In [36]:
def segment_customer(row):
    
    if row['R_score'] == 4 and row['F_score'] == 4 and row['M_score'] == 4:
        return "Champions"
    
    elif row['F_score'] >= 3 and row['M_score'] >= 3:
        return "Loyal Customers"
    
    elif row['R_score'] >= 3 and row['F_score'] >= 2:
        return "Potential Loyalists"
    
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return "At Risk"
    
    else:
        return "Lost Customers"

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [37]:
#Check segment Distribution
rfm['Segment'].value_counts()

Segment
Lost Customers         1769
Loyal Customers        1285
Potential Loyalists     605
Champions               481
At Risk                 199
Name: count, dtype: int64

**Next Step- Export for Power BI**

In [39]:
rfm.to_csv("rfm_customer_segments.csv",index=False)

In [41]:
import os
os.getcwd()

'C:\\Users\\USER'